In [45]:
import io
import zipfile
import requests
import frontmatter

In [46]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [47]:
openclaw_data = read_repo_data('openclaw', 'openclaw')
print(f"OpenClaw documents: {len(openclaw_data)}")

OpenClaw documents: 879


In [48]:
#print(f"openclaw_data_top: {openclaw_data[4:5]}")

In [49]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result


In [50]:
openclaw_chunks_slidingWindow = []

for doc in openclaw_data:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    openclaw_chunks_slidingWindow.extend(chunks)


In [51]:
print(len(openclaw_chunks_slidingWindow))


5788


In [52]:
#print(openclaw_chunks_slidingWindow[2000])

In [53]:
openclaw_chunks= openclaw_chunks_slidingWindow

In [54]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [55]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the Openclaw index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the Openclaw index.
    """
    return openclaw_index.search(query, num_results=5)

def vector_search(query: str) -> List[Any]:
    """
    Perform a semantic search on the Openclaw index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the Openclaw index.
    """
    q = embedding_model.encode(query)
    return openclaw_vindex.search(q, num_results=5)


In [56]:
from minsearch import Index
openclaw_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[]
)

openclaw_index.fit(openclaw_data)


In [57]:
#query = "What is OpenClaw"
#query = "Give me an overview of OpenClaw"
query = "How can I install OpenClaw"
text_search_results = text_search(query)

In [58]:
#print(f'{text_search_results[4]['content']}')
print(len(openclaw_chunks))

5788


In [59]:
## Vector search
from tqdm.auto import tqdm
import numpy as np

openclaw_embeddings = []

for d in tqdm(openclaw_chunks):
    v = embedding_model.encode(d['chunk'])
    openclaw_embeddings.append(v)


  0%|          | 0/5788 [00:00<?, ?it/s]

In [60]:
openclaw_embeddings = np.array(openclaw_embeddings)

In [61]:
from minsearch import VectorSearch
openclaw_vindex = VectorSearch()
openclaw_vindex.fit(openclaw_embeddings, openclaw_chunks)

In [63]:
query = "How can I install OpenClaw"
#query = "Give me an overview of OpenClaw"
vector_search_results = vector_search(query)

In [64]:
system_prompt = """
You are a helpful assistant for a course. 

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
"""

In [66]:
from pydantic_ai import Agent

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
    #tools=[vector_search],
    model='openai:gpt-4o-mini'
#    model='gpt-4o-mini'
)

In [67]:
question = "Give me an overview of OpenClaw"
result_textsearch = await agent.run(user_prompt=question)
result_textsearch

AgentRunResult(output='OpenClaw is a versatile framework designed for integrating chat interfaces and automating interactions across various messaging platforms such as WhatsApp, Telegram, Discord, and more. Below is a summarized overview of its key features and functionality:\n\n### Key Features:\n1. **Multi-Channel Support**: OpenClaw can connect with multiple messaging platforms, allowing seamless interaction across different channels.\n\n2. **Gateway Architecture**: It uses a Gateway, which acts as the core component to manage the connections and operations for different messaging channels.\n\n3. **Command Line Interface (CLI)**: OpenClaw provides a robust CLI for managing the configuration, monitoring status, and troubleshooting. Commands like `openclaw status` and `openclaw logs` are essential for diagnostics and output monitoring.\n\n4. **Logging and Diagnostics**: It offers detailed logging capabilities that can be customized through the CLI or configuration files. Logs can be 

In [69]:
from pydantic_ai import Agent

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    #tools=[text_search],
    tools=[vector_search],
    model='openai:gpt-4o-mini'
#    model='gpt-4o-mini'
)

In [70]:
question = "Give me an overview of OpenClaw"

result_vsearch = await agent.run(user_prompt=question)
result_vsearch 

AgentRunResult(output="OpenClaw appears to be a versatile framework that facilitates the development and integration of AI assistants and other interactive applications. Here's an overview of its key features and functionalities based on the course materials:\n\n### General Overview\n- **Purpose**: OpenClaw is designed to build AI-driven applications, particularly those that can facilitate communication and interaction through various channels.\n- **Community Driven**: It supports community contributions, allowing users to install and create plugins that extend its capabilities.\n\n### Key Features\n- **Plugins**: OpenClaw supports third-party plugins which can enhance its functionality by adding new channels, tools, or providers. These plugins can be easily installed via a command:\n  ```bash\n  openclaw plugins install <package-name>\n  ```\n  Some available plugins include integrations with applications like Codex, DingTalk, and tools for managing context in conversations.\n\n- **En